In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

from sympy import N, Interval, EmptySet, solveset, lambdify


# Phương pháp Dây cung (Secant Method)

# Điều kiện:

* $(a,b)$ là khoảng cách ly nghiệm

* $f'(x)$ xác định dấu không đổi trên $[a,b]$

* $f''(x)$ xác định dấu không đổi trên $[a,b]$

Điều kiện hội tụ:

* Chọn đúng điểm mốc $d$ (điểm Fourier - $f(d) \cdot f''(d) > 0$)
    
* Chọn đúng xuất phát ban đầu $x_0$ ($f(d) \cdot f(x_0) < 0$)

### 1.2 Code
#### 1.2.3 kiểm tra điều kiện hội tụ và kiểm tra đổi dấu của f'(x) và f''(x)

In [ ]:
class Secant_class:
    def __init__(self, expr, a, b, eps):
        self.x = symbols("x")

        self.expr = sympify(expr)
        self.f = lambdify(self.x, self.expr, "numpy")

        self.a = a
        self.b = b
        self.eps = eps

        self.expr_df = diff(self.expr, self.x)
        self.expr_d2f = diff(self.expr, self.x, 2)

        self.f1 = lambdify(self.x, self.expr_df, "numpy")
        self.f2 = lambdify(self.x, self.expr_d2f, "numpy")

        self.rows = []
        self.df = None

    def __no_sign_change(self, expr, a, b):
        """
        Kiểm tra biểu thức expr có không đổi dấu trên khoảng (a,b) hay không.

        Ý tưởng:
        - Giải phương trình expr = 0 trên đoạn [a,b]
        - Nếu có nghiệm nằm trong khoảng mở (a,b) thì expr không giữ dấu nghiêm ngặt
        - Nếu không có nghiệm trong (a,b) thì expr được xem là không đổi dấu trên (a,b)
          với giả thiết expr liên tục trên khoảng xét

        Giá trị trả về:
        - True  : expr không đổi dấu trên (a,b)
        - False : expr có nghiệm trong (a,b)
        - None  : không kết luận được bằng SymPy
        """
        x = self.x
        sol = solveset(expr, x, domain=Interval(a, b))

        if sol.is_FiniteSet:
            for r in sol:
                try:
                    r_val = float(N(r))
                    if a < r_val < b:
                        return False
                except Exception:
                    return None
            return True

        if sol == EmptySet:
            return True

        return None

    def __get_sign_numeric(self, expr, a, b, Nsample=1000, tol=1e-12):
        """
        Kiểm tra dấu gần đúng của expr trên (a,b) bằng cách lấy mẫu số học.

        Giá trị trả về:
        -  1   : expr > 0 trên (a,b) theo kiểm tra số
        - -1   : expr < 0 trên (a,b) theo kiểm tra số
        -  0   : expr đổi dấu hoặc chạm gần 0 trong khoảng
        - None : không kết luận được
        """
        g = lambdify(self.x, expr, "numpy")
        xs = np.linspace(a, b, Nsample + 2)[1:-1]  # bỏ hai đầu mút

        try:
            vals = np.array(g(xs), dtype=float)
        except Exception:
            return None

        if np.any(np.isnan(vals)) or np.any(np.isinf(vals)):
            return None

        if np.any(np.abs(vals) <= tol):
            return 0

        if np.all(vals > 0):
            return 1
        if np.all(vals < 0):
            return -1

        return 0

    def __sign_to_text(self, sign, expr_name, a, b):
        """
        Đổi mã dấu thành chuỗi mô tả dễ đọc.
        """
        if sign == 1:
            return f"{expr_name} > 0 trên ({a}, {b})"
        elif sign == -1:
            return f"{expr_name} < 0 trên ({a}, {b})"
        elif sign == 0:
            return f"{expr_name} không giữ dấu nghiêm ngặt trên ({a}, {b})"
        else:
            return f"Không kết luận chắc chắn được dấu của {expr_name} trên ({a}, {b})"

    def show_convergence_info(self):
        """
        In đầy đủ thông tin kiểm tra điều kiện hội tụ cho phương pháp dây cung.

        Nội dung in ra:
        1. Biểu thức f(x), f'(x), f''(x)
        2. Giá trị f(a), f(b)
        3. Kiểm tra khoảng cách ly nghiệm qua dấu của f(a)*f(b)
        4. Dấu của f'(x) trên (a,b)
        5. Dấu của f''(x) trên (a,b)
        6. Kết luận khoảng có phù hợp điều kiện hội tụ hay không
        """
        a = self.a
        b = self.b
        f = self.f
        f1 = self.f1
        f2 = self.f2

        print("=" * 80)
        print("THÔNG TIN KIỂM TRA ĐIỀU KIỆN HỘI TỤ")
        print("=" * 80)

        # In biểu thức
        print(f"f(x)    = {self.expr}")
        print(f"f'(x)   = {self.expr_df}")
        print(f"f''(x)  = {self.expr_d2f}")
        print("-" * 80)

        # Giá trị tại hai đầu mút
        try:
            fa = f(a)
            fb = f(b)
            print(f"f({a}) = {fa}")
            print(f"f({b}) = {fb}")
        except Exception as e:
            print("Không tính được giá trị của hàm số tại đầu mút.")
            print("Chi tiết:", e)
            return

        print("-" * 80)

        # Kiểm tra khoảng cách ly nghiệm
        if fa == 0:
            print(f"Kết luận: x = {a} là nghiệm đúng.")
            return

        if fb == 0:
            print(f"Kết luận: x = {b} là nghiệm đúng.")
            return

        product = fa * fb
        print(f"f({a}) * f({b}) = {product}")

        if product < 0:
            print(f"[{a}, {b}] là khoảng cách ly nghiệm vì f(a) * f(b) < 0.")
        else:
            print(f"[{a}, {b}] KHÔNG là khoảng cách ly nghiệm vì f(a) * f(b) >= 0.")

        print("-" * 80)

        # Giá trị đạo hàm tại hai đầu mút
        try:
            f1a = f1(a)
            f1b = f1(b)
            f2a = f2(a)
            f2b = f2(b)

            print(f"f'({a})  = {f1a}")
            print(f"f'({b})  = {f1b}")
            print(f"f''({a}) = {f2a}")
            print(f"f''({b}) = {f2b}")
        except Exception as e:
            print("Không tính được đạo hàm tại đầu mút.")
            print("Chi tiết:", e)

        print("-" * 80)

        # Kiểm tra dấu của f'(x)
        sign_df_symbolic = self.__no_sign_change(self.expr_df, a, b)
        sign_df_numeric = self.__get_sign_numeric(self.expr_df, a, b)

        print("Kiểm tra dấu của f'(x):")
        print(f"- Kết quả kiểm tra không đổi dấu bằng SymPy: {sign_df_symbolic}")
        print(f"- Kết quả xác định dấu bằng lấy mẫu số: {sign_df_numeric}")
        print(self.__sign_to_text(sign_df_numeric, "f'(x)", a, b))

        print("-" * 80)

        # Kiểm tra dấu của f''(x)
        sign_d2f_symbolic = self.__no_sign_change(self.expr_d2f, a, b)
        sign_d2f_numeric = self.__get_sign_numeric(self.expr_d2f, a, b)

        print("Kiểm tra dấu của f''(x):")
        print(f"- Kết quả kiểm tra không đổi dấu bằng SymPy: {sign_d2f_symbolic}")
        print(f"- Kết quả xác định dấu bằng lấy mẫu số: {sign_d2f_numeric}")
        print(self.__sign_to_text(sign_d2f_numeric, "f''(x)", a, b))

        print("-" * 80)

        # Kết luận cuối
        ok_interval = (product < 0)
        ok_df = (sign_df_numeric in [1, -1])
        ok_d2f = (sign_d2f_numeric in [1, -1])

        if ok_interval and ok_df and ok_d2f:
            print("KẾT LUẬN:")
            print("Khoảng đã thỏa các điều kiện kiểm tra cơ bản để áp dụng phương pháp dây cung.")
        else:
            print("KẾT LUẬN:")
            print("Khoảng CHƯA thỏa đầy đủ điều kiện kiểm tra cơ bản để áp dụng phương pháp dây cung.")

        print("=" * 80)

    def __check(self, a, b):
        """
        Hàm kiểm tra điều kiện đầu vào để dùng trong thuật toán giải.
        Hàm này gọn, còn phần in chi tiết dùng show_convergence_info().
        """
        f = self.f

        if f(a) == 0:
            print(f"Phương trình có nghiệm đúng x = {a}")
            return a

        if f(b) == 0:
            print(f"Phương trình có nghiệm đúng x = {b}")
            return b

        if f(a) * f(b) >= 0:
            print("Khoảng cách ly không hợp lệ: f(a)*f(b) >= 0")
            return None

        sign_df = self.__get_sign_numeric(self.expr_df, a, b)
        if sign_df == 0:
            print("f'(x) không giữ dấu nghiêm ngặt trên (a,b)")
            return None
        if sign_df is None:
            print("Không kết luận chắc chắn được dấu của f'(x) trên (a,b)")
            return None

        sign_d2f = self.__get_sign_numeric(self.expr_d2f, a, b)
        if sign_d2f == 0:
            print("f''(x) không giữ dấu nghiêm ngặt trên (a,b)")
            return None
        if sign_d2f is None:
            print("Không kết luận chắc chắn được dấu của f''(x) trên (a,b)")
            return None

        return True

#### 1.2.2 Có cộng thêm sai số làm tròn
##### Công thức 1: sai số mục tiêu - sai số tuyệt đối

In [ ]:
# dây cung theo sai số mục tiêu - sai số tuyệt đối
# có cộng thêm sai số làm tròn

def secant_iteration_v1 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a
	mrk = b

	results = []
	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = abs(f(x_new))/ m1

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=|f(x_n)| / m_1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v1 (f, df, a, b, n, rbl)

##### Công thức 2: sai số theo hai lần xấp xỉ - tuyệt đối

In [ ]:
# dây cung theo sai số hai lần xấp xỉ có cộng - sai số tuyệt đối
# có cộng thêm sai số làm tròn
def secant_iteration_v2 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0;

	# Implementing Secant Method
	x = a; mrk = b;

	results = [];
	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = (M1 - m1) * abs(x - x_new) / m1

		results.append({
            'n': i,
			'mrk': mrk,
            'x_(n+1)': x_new,
            'f(x_(n+1))': f(x_new),
            'delta_x=(M1 - m1)*(x-x_new)/m1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v2 (f, df, a, b, n, rbl)